In [ ]:
import warnings
warnings.filterwarnings('ignore')

import kagglehub
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import KNNImputer
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_selection import SelectKBest, f_classif, chi2
from sklearn.metrics import (confusion_matrix, classification_report,
                           accuracy_score, precision_score, recall_score,
                           f1_score, make_scorer)
from imblearn.over_sampling import SMOTE
import matplotlib.colors as mcolors
from sklearn.model_selection import learning_curve

In [ ]:

aimlveera_counterfeit_product_detection_dataset_path = kagglehub.dataset_download('aimlveera/counterfeit-product-detection-dataset')
products_df = pd.read_csv(os.path.join(aimlveera_counterfeit_product_detection_dataset_path, "counterfeit_products.csv"), encoding='utf-8')
transactions_df = pd.read_csv(os.path.join(aimlveera_counterfeit_product_detection_dataset_path, "_counterfeit_transactions.csv"), encoding='ascii')

print('Products dataset shape:', products_df.shape)
print('Transactions dataset shape:', transactions_df.shape)


In [ ]:
# Làm sạch tên cột cho cả 2 dataset
products_df.columns = products_df.columns.str.strip()
transactions_df.columns = transactions_df.columns.str.strip()

In [ ]:
# Xem 5 dòng đầu của data transaction
print('Dữ liệu data transaction')
transactions_df.head()


In [ ]:
transactions_df.info()

In [ ]:
transactions_df.describe()

In [ ]:
#Xem 5 dòng đầu của data product
products_df.head()

In [ ]:
products_df.info()

In [ ]:
products_df.describe()

**EDA Bộ dữ liệu Transaction**

In [ ]:
# Đồ thị phân bổ tỷ lệ phần trăm giao dịch có liên quan hàng giả
counterfeit_counts = transactions_df['involves_counterfeit'].value_counts()
counterfeit_percentages = counterfeit_counts / len(transactions_df) * 100

labels = ['0 - Giao dịch hàng thật', '1 - Giao dịch hàng giả']

plt.figure(figsize=(8, 8))
wedges, texts, autotexts = plt.pie(
    counterfeit_percentages,
    autopct='%1.1f%%',
    colors=['#F2B6CC', '#F24976'],
    startangle=90
)

for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')
    autotext.set_fontsize(14)

plt.legend(wedges, labels,
          title="Loại giao dịch",
          loc="center left",
          bbox_to_anchor=(1, 0, 0.5, 1),
          fontsize=11,
          title_fontsize=12)

plt.title('Đồ thị phân bổ tỷ lệ phần trăm giao dịch có liên quan hàng giả',
          fontsize=13, fontweight='bold', pad=20)

plt.axis('equal')
plt.tight_layout()
plt.show()

In [ ]:
transaction_analysis_cols_hist = [
    'unit_price', 'quantity', 'total_amount', 'customer_age',
    'shipping_cost', 'delivery_time_days',
    'customer_history_orders'
]

numeric_transaction_analysis_cols = transactions_df[transaction_analysis_cols_hist].select_dtypes(include=np.number).columns.tolist()

for col in numeric_transaction_analysis_cols:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Tạo biểu đồ histogram
    sns.histplot(transactions_df[col], kde=True, color='#F2B6CC',
                 line_kws={'color': '#F24976', 'linewidth': 2},
                 ax=axes[0])
    axes[0].set_title(f'Biểu đồ histogram cho {col}', fontsize=13, fontweight='bold')
    axes[0].set_xlabel(col, fontsize=11, fontweight='bold')
    axes[0].set_ylabel('Tần suất', fontsize=11, fontweight='bold')
    axes[0].grid(axis='y', alpha=0.3)

    # Tạo biểu đồ Box plot
    sns.boxplot(x=transactions_df[col], color='#F2B6CC',
                linewidth=2, ax=axes[1])
    axes[1].set_title(f'Biểu đồ boxplot cho {col}', fontsize=13, fontweight='bold')
    axes[1].set_xlabel(col, fontsize=11, fontweight='bold')
    axes[1].grid(axis='x', alpha=0.3)

    fig.suptitle(f'EDA cho: {col}',
                 fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()


In [ ]:
#Biểu đồ countplot hiển thị phân phối tần suất của các phương thức thanh toán
plt.figure(figsize=(10, 6))
sns.countplot(y='payment_method', data=transactions_df, order = transactions_df['payment_method'].value_counts().index, palette='RdPu')
plt.title('Danh mục các phương thức thanh toán')
plt.xlabel('Số lượng giao dịch')
plt.ylabel('Phương thức thanh toán')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(y='customer_location', data=transactions_df, order = transactions_df['customer_location'].value_counts().index, palette='RdPu')
plt.title('Phân phối địa điểm khách hàng')
plt.xlabel('Số lượng giao dịch')
plt.ylabel('Địa điểm khách hàng')
plt.show()

In [ ]:
#Heatmap cho Transaction
numerical_data =transactions_df.select_dtypes(include=['int64', 'float64', 'bool'])

for col in numerical_data.select_dtypes(include=['bool']).columns:
    numerical_data[col] = numerical_data[col].astype(int)

correlation_matrix = numerical_data.corr()

colors = ["#F24976", "#FFF5F2", "#F2B6CC"]
cmap_custom = mcolors.LinearSegmentedColormap.from_list("my_pink_beige_cmap", colors)

plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, cmap=cmap_custom, fmt=".2f")
plt.title('Ma trận tương quan các biến num trong Transaction Dataset')
plt.show()

print("Bảng tổng hợp tương quan với involves_counterfeit:")
print(correlation_matrix['involves_counterfeit'].sort_values(ascending=False))

In [ ]:
# Vẽ pairplot cho các biến tương quan cao
cols_for_pairplot = ['quantity', 'customer_history_orders', 'refund_requested', 'involves_counterfeit', 'total_amount','unit_price']
pairplot_data = transactions_df[cols_for_pairplot]

pink_palette = ['#F2B5CC', '#F24976']
sns.pairplot(pairplot_data, hue='involves_counterfeit', diag_kind='kde', palette=pink_palette)
plt.suptitle('Biểu đồ Pairplot cho các biến tương quan cao', y=1.05)
plt.show()

In [ ]:
# Biểu đồ thời gian cho phần trăm các giao dịch giả theo tháng
transactions_df['transaction_date'] = pd.to_datetime(transactions_df['transaction_date'])
transactions_df['transaction_month'] = transactions_df['transaction_date'].dt.to_period('M')
fraud_rate_over_time = transactions_df.groupby('transaction_month')['involves_counterfeit'].value_counts(normalize=True).unstack().fillna(0)

plt.figure(figsize=(12, 6))
fraud_rate_over_time[True].plot(kind='line', color='#F24976')
plt.title('Tỷ lệ phần trăm giao dịch giả theo tháng')
plt.xlabel('Tháng')
plt.ylabel('Phần trăm')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
#Biểu đồ cột chồng các cờ cảnh báo
flag_cols = ['velocity_flag', 'geolocation_mismatch', 'device_fingerprint_new', 'refund_requested']
data_flags =transactions_df[flag_cols + ['involves_counterfeit']]

for col in flag_cols:
    data_flags[col] = data_flags[col].astype(int)
flag_proportions = data_flags.groupby('involves_counterfeit')[flag_cols].mean().T
flag_proportions.columns = ['GD Hàng thật', 'GD Hàng giả']

# Create the stacked bar chart
flag_proportions.plot(kind='bar', stacked=True, figsize=(10, 6), color=['#F2B6CC', '#F24976'])

plt.title('Biểu đồ cột chồng các cờ cảnh báo')
plt.xlabel('Cờ cảnh báo')
plt.ylabel('Tỷ lệ')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Chú thích')
plt.tight_layout()
plt.show()

# EDA cho **Product**

In [ ]:
#Biểu đồ tròn biểu thị tỷ lệ phần trăm sản phẩm thật giả trong dataset
counterfeit_product_counts = products_df['is_counterfeit'].value_counts()
counterfeit_product_percentages = counterfeit_product_counts / len(products_df) * 100
labels = ['0 - Hàng thật', '1 - Hàng giả']

plt.figure(figsize=(8, 8))
wedges, texts, autotexts = plt.pie(
    counterfeit_product_percentages,
    autopct='%1.1f%%',
    colors=['#F2B6CC', '#F24976'],
    startangle=90
)

for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')
    autotext.set_fontsize(14)

plt.legend(wedges, labels,
          title="Chú thích",
          loc="center left",
          bbox_to_anchor=(1, 0, 0.5, 1),
          fontsize=11,
          title_fontsize=12)

plt.title('Đồ thị phân bổ tỷ lệ phần trăm hàng giả',
          fontsize=13, fontweight='bold', pad=20)

plt.axis('equal')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12,6))
sns.countplot(data=products_df, y='category', order=products_df['category'].value_counts().index, palette='RdPu')
plt.title('Tỷ trọng danh mục phân loại hàng')
plt.xlabel('Tần suất')
plt.ylabel('Danh mục')
plt.tight_layout()
plt.show()

In [ ]:
#EDA cho các đặc trưng
product_analysis_cols_hist = [
    'product_images', 'spelling_errors', 'payment_methods_count',
    'contact_info_complete', 'return_policy_clear', 'price',
    'seller_rating', 'seller_reviews', 'description_length',
    'shipping_time_days', 'domain_age_days', 'views',
    'purchases', 'warranty_months',
]

numeric_product_analysis_cols = products_df[product_analysis_cols_hist].select_dtypes(include=np.number).columns.tolist()

for col in numeric_product_analysis_cols:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    #Histogram cho các đặc trưng
    sns.histplot(products_df[col], kde=True, color='#F2B6CC',
                 line_kws={'color': '#F24976', 'linewidth': 2},
                 ax=axes[0])
    axes[0].set_title(f'Biểu đồ Histogram cho {col}', fontsize=13, fontweight='bold')
    axes[0].set_xlabel(col, fontsize=11, fontweight='bold')
    axes[0].set_ylabel('Frequency', fontsize=11, fontweight='bold')
    axes[0].grid(axis='y', alpha=0.3)
    # Boxplot cho các đặc trưng
    sns.boxplot(x=products_df[col], color='#F2B6CC',
                linewidth=2, ax=axes[1])
    axes[1].set_title(f'Biểu đồ Boxplot cho {col}', fontsize=13, fontweight='bold')
    axes[1].set_xlabel(col, fontsize=11, fontweight='bold')
    axes[1].grid(axis='x', alpha=0.3)

    fig.suptitle(f'EDA cho: {col}',
                 fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:

product_corr_cols = [
    'product_images', 'spelling_errors', 'payment_methods_count',
    'contact_info_complete', 'return_policy_clear', 'price',
    'seller_rating', 'seller_reviews', 'description_length',
    'shipping_time_days', 'domain_age_days', 'views',
    'purchases', 'certification_badges', 'warranty_months',
    'bulk_orders', 'unusual_payment_patterns', 'is_counterfeit'
]

product_corr_data = products_df[product_corr_cols].copy()

for col in ['contact_info_complete', 'return_policy_clear', 'bulk_orders', 'unusual_payment_patterns', 'is_counterfeit']:
    if col in product_corr_data.columns:
        product_corr_data[col] = product_corr_data[col].astype(int)
product_correlation_matrix = product_corr_data.corr()

colors = ["#F24976", "#FFF5F2", "#F2B6CC"]
cmap_custom = mcolors.LinearSegmentedColormap.from_list("my_pink_beige_cmap", colors)

plt.figure(figsize=(12, 10))
sns.heatmap(product_correlation_matrix, annot=True, cmap=cmap_custom, fmt=".2f")
plt.title('Correlation Matrix of Selected Product Features')
plt.show()

print("Tương quan với 'is_counterfeit':")
print(product_correlation_matrix['is_counterfeit'].sort_values(ascending=False))
pairplot_specific_cols= ['shipping_time_days', 'spelling_errors', 'return_policy_clear','price','contact_info_complete','seller_reviews','domain_age_days',
'description_length','product_images','seller_rating','payment_methods_count' ]

In [ ]:
# Vẽ pairplot các biến tương quan cao
pairplot_data_full = products_df[pairplot_specific_cols + ['is_counterfeit']].copy()

boolean_cols_to_convert = ['return_policy_clear', 'contact_info_complete']
for col in boolean_cols_to_convert:
    if col in pairplot_data_full.columns:
        pairplot_data_full[col] = pairplot_data_full[col].astype(int)

pink_palette_hue = {False: '#F2B6CC', True: '#F24976'}

# Tách ra thành 4 nhóm: Saler Features
print(" Biểu đồ Pair Plot - Seller Features")
seller_features = ['seller_rating', 'seller_reviews', 'domain_age_days', 'shipping_time_days', 'is_counterfeit']
sns.pairplot(pairplot_data_full[seller_features],
             hue='is_counterfeit',
             palette=pink_palette_hue,
             diag_kind='kde',
             plot_kws={'alpha': 0.6, 's': 30},
             diag_kws={'alpha': 0.7})
plt.suptitle('Biểu đồ Pair Plot cho các Seller Features có tương quan cao', y=1.00, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Tách ra thành 4 nhóm: Product Features
print("Biểu đồ Pair Plot -Product Features")
product_features = ['price', 'description_length', 'product_images', 'spelling_errors', 'is_counterfeit']
sns.pairplot(pairplot_data_full[product_features],
             hue='is_counterfeit',
             palette=pink_palette_hue,
             diag_kind='kde',
             plot_kws={'alpha': 0.6, 's': 30},
             diag_kws={'alpha': 0.7})
plt.suptitle('Biểu đồ Pair Plot cho các Product Features', y=1.00, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

#Tách ra thành 4 nhóm: Website Features
print("Biểu đồ Pair Plot - Website Features")
website_features = ['payment_methods_count', 'contact_info_complete', 'return_policy_clear', 'is_counterfeit']
sns.pairplot(pairplot_data_full[website_features],
             hue='is_counterfeit',
             palette=pink_palette_hue,
             diag_kind='kde',
             plot_kws={'alpha': 0.6, 's': 30},
             diag_kws={'alpha': 0.7})
plt.suptitle('Biểu đồ Pair Plot cho các Website Features', y=1.00, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Select Feature cho Transaction**

In [ ]:
# Chuẩn bị dữ liệu cho dataset transactions
exclude_cols = ['transaction_id', 'customer_id', 'transaction_date', 'transaction_month', 'involves_counterfeit'] #Các cột có typedata không thích hợp hoặc không cần thiết hoặc Cột mục tiêu
X_trans = transactions_df.drop(columns=exclude_cols, errors='ignore')
y_trans = transactions_df['involves_counterfeit']

# Xác định features số và phân loại
numeric_features_trans = []
categorical_features_trans = []

for col in X_trans.columns:
    if X_trans[col].dtype in ['int64', 'float64']:
        if X_trans[col].nunique() <= 20 and X_trans[col].min() >= 0:
            categorical_features_trans.append(col)
        else:
            numeric_features_trans.append(col)
    elif X_trans[col].dtype == 'bool':
        categorical_features_trans.append(col)
    else:  # object type
        categorical_features_trans.append(col)

print(f"\nNumeric features ({len(numeric_features_trans)}):")
for feature in numeric_features_trans:print("-", feature)
print(f"\nCategorical features ({len(categorical_features_trans)}):")
for feature in categorical_features_trans:print("-", feature)


In [ ]:
# Kiểm định Anova cho Transactions
if len(numeric_features_trans) > 0:
    print("Kiểm định cho biến num")
    X_numeric_trans = X_trans[numeric_features_trans].copy()

    # Thực hiện ANOVA test
    selector_anova_trans = SelectKBest(score_func=f_classif, k='all')
    selector_anova_trans.fit(X_numeric_trans, y_trans)

    # Tạo bảng kết quả ANOVA
    anova_results_trans = pd.DataFrame({
        'Đặc trưng': numeric_features_trans,
        'F_score': selector_anova_trans.scores_,
        'P_value': selector_anova_trans.pvalues_,
        'Giả thuyết': selector_anova_trans.pvalues_ < 0.05
    }).sort_values('P_value')

    print("Kết quả ANOVA cho Transactions (top 10):")
    print(anova_results_trans.head(10).round(4))


In [ ]:
# Kiểm định Chi-square cho Transaction
if len(categorical_features_trans) > 0:
    print("Kiểm định Chi-square cho biến phân loại")
    X_categorical_trans = X_trans[categorical_features_trans].copy()
    le_trans = LabelEncoder()
    # Xử lý dữ liệu cho biến phân loại
    for col in categorical_features_trans:
        if X_categorical_trans[col].dtype == 'object':
            X_categorical_trans[col] = X_categorical_trans[col].fillna('missing')
            X_categorical_trans[col] = le_trans.fit_transform(X_categorical_trans[col].astype(str))
        elif X_categorical_trans[col].dtype == 'bool':
            X_categorical_trans[col] = X_categorical_trans[col].astype(int)
        else:
            mode_val = X_categorical_trans[col].mode()
            fill_val = mode_val.iloc[0] if not mode_val.empty else 0
            X_categorical_trans[col] = X_categorical_trans[col].fillna(fill_val)
    X_categorical_trans = X_categorical_trans.astype(int)
    X_categorical_trans = X_categorical_trans.abs()
    # Thực hiện Chi-square test
    selector_chi2_trans = SelectKBest(score_func=chi2, k='all')
    selector_chi2_trans.fit(X_categorical_trans, y_trans)
    # Tạo bảng kết quả Chi-square
    chi2_results_trans = pd.DataFrame({
        'Đặc trưng': categorical_features_trans,
        'Chi2_score': selector_chi2_trans.scores_,
        'P_value': selector_chi2_trans.pvalues_,
        'Giả thuyết': selector_chi2_trans.pvalues_ < 0.05,
        'Số phân loại': [X_trans[col].nunique() for col in categorical_features_trans]
    }).sort_values('P_value')
    print("Kết quả Chi-square cho Transactions:")
    print(chi2_results_trans.round(4))

# **Feature Selection cho Product**

In [ ]:

X_prod = products_df.drop('is_counterfeit', axis=1)
y_prod = products_df['is_counterfeit']

numeric_features_prod = []
categorical_features_prod = []

for col in X_prod.columns:
    if X_prod[col].dtype in ['int64', 'float64']:
        if X_prod[col].nunique() <= 20 and X_prod[col].min() >= 0:
            categorical_features_prod.append(col)
        else:
            numeric_features_prod.append(col)
    elif X_prod[col].dtype == 'bool':
        categorical_features_prod.append(col)
    else:  # object type
        categorical_features_prod.append(col)

print(f"\nNumeric features ({len(numeric_features_prod)}): {numeric_features_prod}")
print(f"Categorical features ({len(categorical_features_prod)}): {categorical_features_prod}")


In [ ]:
# Phân tích Anova cho Dataset Product

if len(numeric_features_prod) > 0:

    X_numeric_prod = X_prod[numeric_features_prod].copy()
    X_numeric_prod = X_numeric_prod.fillna(X_numeric_prod.mean())

    selector_anova_prod = SelectKBest(score_func=f_classif, k='all')
    selector_anova_prod.fit(X_numeric_prod, y_prod)

    anova_results_prod = pd.DataFrame({
        'Đặc trưng': numeric_features_prod,
        'F_score': selector_anova_prod.scores_,
        'P_value': selector_anova_prod.pvalues_,
        'Giả thuyết': selector_anova_prod.pvalues_ < 0.05
    }).sort_values('P_value')

    print("Kết quả ANOVA cho Products (top 10):")
    print(anova_results_prod.head(10).round(4))


In [ ]:
# Phân tích Chi-square cho Dataset Product
if len(categorical_features_prod) > 0:
    X_categorical_prod = X_prod[categorical_features_prod].copy()
    le_prod = LabelEncoder()
    for col in categorical_features_prod:
        if X_categorical_prod[col].dtype == 'object':
            X_categorical_prod[col] = X_categorical_prod[col].fillna('missing')
            X_categorical_prod[col] = le_prod.fit_transform(X_categorical_prod[col].astype(str))
        elif X_categorical_prod[col].dtype == 'bool':
            X_categorical_prod[col] = X_categorical_prod[col].astype(int)
        else:
            mode_val = X_categorical_prod[col].mode()
            fill_val = mode_val.iloc[0] if not mode_val.empty else 0
            X_categorical_prod[col] = X_categorical_prod[col].fillna(fill_val)
    X_categorical_prod = X_categorical_prod.astype(int)
    X_categorical_prod = X_categorical_prod.abs()
    selector_chi2_prod = SelectKBest(score_func=chi2, k='all')
    selector_chi2_prod.fit(X_categorical_prod, y_prod)
    chi2_results_prod = pd.DataFrame({
        'Đặc trưng': categorical_features_prod,
        'Chi2_score': selector_chi2_prod.scores_,
        'P_value': selector_chi2_prod.pvalues_,
        'Giả thuyết': selector_chi2_prod.pvalues_ < 0.05,
        'Số phân loại': [X_prod[col].nunique() for col in categorical_features_prod]
    }).sort_values('P_value')
    print("Kết quả Chi-square cho Products:")
    print(chi2_results_prod.round(4))


***# Lựa chọn của em là***

In [ ]:

print("\n=== Tổng hợp đặc trưng được chọn ===")
selected_features = []

# Thêm top numeric features
if 'anova_results_trans' in locals():
    selected_features.extend(anova_results_trans[anova_results_trans['P_value'] < 0.01]['Đặc trưng'].tolist())
if 'anova_results_prod' in locals():
    selected_features.extend(anova_results_prod[anova_results_prod['P_value'] < 0.01]['Đặc trưng'].tolist())

# Thêm top categorical features
if 'chi2_results_trans' in locals():
    selected_features.extend(chi2_results_trans[chi2_results_trans['P_value'] < 0.01]['Đặc trưng'].tolist())
if 'chi2_results_prod' in locals():
    selected_features.extend(chi2_results_prod[chi2_results_prod['P_value'] < 0.01]['Đặc trưng'].tolist())

# Loại bỏ trùng lặp
selected_features = list(set(selected_features))

print(f"Tổng số features được chọn: {len(selected_features)}")
print("Danh sách features:")
for i, feature in enumerate(selected_features, 1):
    print(f"{i:2d}. {feature}")



In [ ]:
features_to_remove = ['product_id', 'seller_id', 'brand', 'listing_date']
selected_features = [feature for feature in selected_features if feature not in features_to_remove]

product_features = []
transaction_features = []

print("Phân bổ Features đã chọn (sau khi loại bỏ):")
print("--- Products Dataset ---")
for feature in selected_features:
    if feature in products_df.columns:
        print(f"{feature}")
        product_features.append(feature)

print("\n--- Transactions Dataset ---")
for feature in selected_features:
    if feature in transactions_df.columns:
        print(f"{feature}")
        transaction_features.append(feature)

# **Xử lý tiền dữ liệu**

***Làm sạch dữ liệu***

In [ ]:

print("TRANSACTIONS DATASET:")
transaction_features_df = transactions_df[transaction_features].copy()

imputer = KNNImputer(n_neighbors=5)

# Xử lý boolean columns
bool_cols_for_imputation = transaction_features_df.select_dtypes(include='bool').columns
if len(bool_cols_for_imputation) > 0:
    transaction_features_df[bool_cols_for_imputation] = transaction_features_df[bool_cols_for_imputation].astype(int)

# Kiểm tra và xử lý missing values
if transaction_features_df.isnull().sum().sum() > 0:
    print("Phát hiện giá trị thiếu trong dữ liệu.")
    numeric_cols_for_imputation = transaction_features_df.select_dtypes(include=['int64', 'float64']).columns
    if len(numeric_cols_for_imputation) > 0:
        transaction_features_df[numeric_cols_for_imputation] = imputer.fit_transform(
            transaction_features_df[numeric_cols_for_imputation]
        )
        print(f"Đã điền giá trị thiếu cho {len(numeric_cols_for_imputation)} cột numeric")
else:
    print("Không phát hiện missing value")


print("PRODUCTS DATASET:")
product_features_df = products_df[product_features].copy()

# Xử lý boolean columns
bool_cols_for_imputation = product_features_df.select_dtypes(include='bool').columns
if len(bool_cols_for_imputation) > 0:
    product_features_df[bool_cols_for_imputation] = product_features_df[bool_cols_for_imputation].astype(int)

# Kiểm tra và xử lý missing values
if product_features_df.isnull().sum().sum() > 0:
    print("Phát hiện giá trị thiếu trong dữ liệu.")
    numeric_cols_for_imputation = product_features_df.select_dtypes(include=['int64', 'float64']).columns
    if len(numeric_cols_for_imputation) > 0:
        product_features_df[numeric_cols_for_imputation] = imputer.fit_transform(
            product_features_df[numeric_cols_for_imputation]
        )
        print(f"Đã điền giá trị thiếu cho {len(numeric_cols_for_imputation)} cột numeric")
else:
    print("Không phát hiện missing value")

In [ ]:
# Kiểm tra dữ liệu trùng lặp
if transaction_features_df.duplicated().sum() > 0:
    initial_rows = len(transaction_features_df)
    transaction_features_df = transaction_features_df.drop_duplicates(keep='first')
    print(f"Đã loại bỏ {initial_rows - len(transaction_features_df)} dòng dữ liệu trùng lặp.")
else:
    print("Không có dòng dữ liệu trùng lặp trong Dataset transactions.")

if product_features_df.duplicated().sum() > 0:
    initial_rows = len(product_features_df)
    product_features_df = product_features_df.drop_duplicates(keep='first')
    print(f"Đã loại bỏ {initial_rows - len(product_features_df)} dòng dữ liệu trùng lặp.")
else:
    print("Không có dòng dữ liệu trùng lặp trong Dataset Products.")


## ***PHÁT HIỆN SẢN PHẨM GIẢ (PRODUCTS)***

## **Bước 1: Training Models(trên tập Train)**

In [ ]:
# Chuẩn bị dữ liệu products
X_products = product_features_df[product_features].copy()
y_products = products_df['is_counterfeit'].copy()

# Xử lý categorical và boolean
categorical_cols_prod = X_products.select_dtypes(include='object').columns
if len(categorical_cols_prod) > 0:
    X_products = pd.get_dummies(X_products, columns=categorical_cols_prod, drop_first=True)

bool_cols_prod = X_products.select_dtypes(include='bool').columns
if len(bool_cols_prod) > 0:
    X_products[bool_cols_prod] = X_products[bool_cols_prod].astype(int)

# Chia dữ liệu
X_temp_prod, X_test_prod, y_temp_prod, y_test_prod = train_test_split(
    X_products, y_products, test_size=0.1, random_state=42, stratify=y_products
)

X_train_prod, X_val_prod, y_train_prod, y_val_prod = train_test_split(
    X_temp_prod, y_temp_prod, test_size=1/9, random_state=42, stratify=y_temp_prod
)
# BƯỚC 1: Học giới hạn CHỈ TỪ TRAIN SET (tránh data leakage)
def fit_outlier_detector(df, columns, factor=1.5):
    bounds = {}
    for col in columns:
        if col in df.columns and df[col].dtype in ['int64', 'float64']:
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            bounds[col] = {
                'lower': Q1 - factor * IQR,
                'upper': Q3 + factor * IQR
            }
    return bounds
# Chuyển các giá trị outliers về bằng lower hoặc upper
def transform_outliers(df, bounds):
    df = df.copy()
    for col, bound in bounds.items():
        if col in df.columns:
            df[col] = df[col].clip(bound['lower'], bound['upper'])
    return df

# Xác định numeric columns cần xử lý và tính giới hạn
numeric_cols_prod = X_train_prod.select_dtypes(include=['int64', 'float64']).columns.tolist()
outlier_bounds_prod = fit_outlier_detector(X_train_prod, numeric_cols_prod)

# BƯỚC 2: Áp dụng cho tất cả các tập
X_train_prod = transform_outliers(X_train_prod, outlier_bounds_prod)
X_val_prod = transform_outliers(X_val_prod, outlier_bounds_prod)
X_test_prod = transform_outliers(X_test_prod, outlier_bounds_prod)

print(f"Đã xử lý outliers cho {len(outlier_bounds_prod)} features")

# Scale và SMOTE
scaler_prod = StandardScaler()
X_train_prod_scaled = scaler_prod.fit_transform(X_train_prod)
X_val_prod_scaled = scaler_prod.transform(X_val_prod)
X_test_prod_scaled = scaler_prod.transform(X_test_prod)

smote_prod = SMOTE(random_state=42)
X_train_prod_balanced, y_train_prod_balanced = smote_prod.fit_resample(
    X_train_prod_scaled, y_train_prod
)

print(f"Products - Train: {X_train_prod.shape}, Val: {X_val_prod.shape}, Test: {X_test_prod.shape}")
print(f"Train distribution: {pd.Series(y_train_prod).value_counts().to_dict()}")
print(f"Val distribution: {pd.Series(y_val_prod).value_counts().to_dict()}")
print(f"Test distribution: {pd.Series(y_test_prod).value_counts().to_dict()}")


print(f"Sau SMOTE - tệp train Products: {pd.Series(y_train_prod_balanced).value_counts().to_dict()}")
gnb_param_grid_prod = {
    'var_smoothing': np.logspace(-12, -8, 20),
    'priors': [
        None,  # Tự động từ dữ liệu
        [0.3, 0.7],  # Giả định 30% counterfeit
        [0.4, 0.6],
        [0.5, 0.5],  # Equal prior
        [0.6, 0.4],
        [0.7, 0.3]
    ]
}

gnb_grid_prod = GridSearchCV(
    estimator=GaussianNB(),
    param_grid=gnb_param_grid_prod,
    scoring='recall',
    cv=5,
    n_jobs=-1,
    verbose=1
)

#GridSearchCV (5-fold CV)
gnb_grid_prod.fit(X_train_prod_balanced, y_train_prod_balanced)

# Lưu model tốt nhất
best_gnb_prod = gnb_grid_prod.best_estimator_

# ===== 1.2 DECISION TREE =====

dt_param_grid_prod = {
    'max_depth': [5, 10, 15, 20, 25, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 4, 8],
    'criterion': ['gini', 'entropy']
}

dt_grid_prod = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42, criterion="gini"),
    param_grid=dt_param_grid_prod,
    scoring='recall',
    cv=5,
    n_jobs=-1,
    verbose=1
)

#Training Decision Tree với GridSearchCV (5-fold CV)
dt_grid_prod.fit(X_train_prod_balanced, y_train_prod_balanced)

# Lưu model tốt nhất
best_dt_prod = dt_grid_prod.best_estimator_

## **Bước 2: Phân tích Hyperparameters trên Validation Set**

In [ ]:
# ===== 2.1 GAUSSIAN NB - Var Smoothing Analysis =====

var_smoothing_values = np.logspace(-12, -8, 20)
# Đảm bảo có chính xác giá trị 1e-8
if 1e-8 not in var_smoothing_values:
    var_smoothing_values = np.append(var_smoothing_values, 1e-8)
    var_smoothing_values = np.sort(var_smoothing_values)

gnb_val_results = []

for vs in var_smoothing_values:
    gnb_temp = GaussianNB(var_smoothing=vs, priors=gnb_grid_prod.best_params_['priors'])
    gnb_temp.fit(X_train_prod_balanced, y_train_prod_balanced)

    y_val_pred_temp = gnb_temp.predict(X_val_prod_scaled)
    val_recall = recall_score(y_val_prod, y_val_pred_temp)

    gnb_val_results.append({
        'var_smoothing': vs,
        'val_recall': val_recall
    })

gnb_val_df = pd.DataFrame(gnb_val_results)

# ===== CỐ ĐỊNH GIÁ TRỊ TỐI ƯU TẠI 1e-8 =====
best_var_smoothing = 1e-8

# Tìm chỉ số có giá trị tốt nhất
best_idx = (gnb_val_df['var_smoothing'] - best_var_smoothing).abs().idxmin()
actual_best_value = gnb_val_df.loc[best_idx, 'var_smoothing']

print(f"Target var_smoothing: {best_var_smoothing:.2e}")
print(f"   Actual value used: {actual_best_value:.2e}")
print(f"   Index: {best_idx}")

# Visualize
plt.figure(figsize=(12, 5))
plt.plot(gnb_val_df['var_smoothing'], gnb_val_df['val_recall'],
         marker='o', color='#F24976', linewidth=2, markersize=6)
plt.xscale('log')
plt.xlabel('Var Smoothing', fontsize=12, fontweight='bold')
plt.ylabel('Recall on Validation Set', fontsize=12, fontweight='bold')
plt.title('Gaussian NB: Var Smoothing vs Validation Recall (Products)',
          fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)

# Vẽ đường kẻ đứt tại best var smoothing
plt.axvline(x=best_var_smoothing,
            color='#F2B6CC', linestyle='--', linewidth=2, label='Best Param')

# Annotate giá trị tại best param
best_recall = gnb_val_df.loc[best_idx, 'val_recall']
plt.text(best_var_smoothing, best_recall,
         f'{best_recall:.3f}',
         fontsize=10, ha='left', va='bottom', color='#F24976', fontweight='bold')

plt.legend()
plt.tight_layout()
plt.show()

# ===== 2.2 GAUSSIAN NB - Priors Analysis =====
priors_list = [
    (None, 'Auto'),
    ([0.3, 0.7], '0.3/0.7'),
    ([0.4, 0.6], '0.4/0.6'),
    ([0.5, 0.5], '0.5/0.5'),
    ([0.6, 0.4], '0.6/0.4'),
    ([0.7, 0.3], '0.7/0.3')
]

gnb_priors_results = []

# Sử dụng best_var_smoothing đã tìm được
for prior_val, prior_label in priors_list:
    gnb_temp = GaussianNB(var_smoothing=best_var_smoothing,
                          priors=prior_val)
    gnb_temp.fit(X_train_prod_balanced, y_train_prod_balanced)

    y_val_pred_temp = gnb_temp.predict(X_val_prod_scaled)
    val_recall = recall_score(y_val_prod, y_val_pred_temp)

    gnb_priors_results.append({
        'priors': prior_label,
        'val_recall': val_recall
    })

gnb_priors_df = pd.DataFrame(gnb_priors_results)

#Tìm priors tối ưu
best_prior_idx = gnb_priors_df['val_recall'].idxmax()
best_prior_label = gnb_priors_df.loc[best_prior_idx, 'priors']
best_prior_recall = gnb_priors_df.loc[best_prior_idx, 'val_recall'] # Calculate best_prior_recall here

# Visualize
plt.figure(figsize=(10, 6))
bars = plt.bar(gnb_priors_df['priors'], gnb_priors_df['val_recall'],
               color='#F2B6CC', edgecolor='#F24976', linewidth=2)
for i, (bar, prior) in enumerate(zip(bars, gnb_priors_df['priors'])):
    if prior == best_prior_label:
        bar.set_color('#F24976')
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.3f}',
             ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.xlabel('Priors (Genuine/Counterfeit)', fontsize=12, fontweight='bold')
plt.ylabel('Recall on Validation Set', fontsize=12, fontweight='bold')
plt.title('Gaussian NB: Priors vs Validation Recall (Products)',
          fontsize=14, fontweight='bold')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.ylim([0, 1.1])
plt.tight_layout()
plt.show()


print("Tổng kết")
summary_data_prod = {
    'Hyperparameter': ['Var Smoothing', 'Priors'],
    'Best Value': [f"{best_var_smoothing:.2e}", best_prior_label],
    'Validation Recall': [best_recall, best_prior_recall]
}

summary_df_prod = pd.DataFrame(summary_data_prod)
print(summary_df_prod.to_string(index=False))

In [ ]:

print("Lựa chọn siêu tham số cho DT_Products")

cv_results_df = pd.DataFrame(dt_grid_prod.cv_results_)

cv_results_df['max_depth_str'] = cv_results_df['param_max_depth'].apply(
    lambda x: 'None' if x is None else str(x)
)
cv_results_df['min_samples_split_str'] = cv_results_df['param_min_samples_split'].astype(str)
cv_results_df['min_samples_leaf_str'] = cv_results_df['param_min_samples_leaf'].astype(str)
cv_results_df['criterion_str'] = cv_results_df['param_criterion'].astype(str)

#Đánh giá 4 metrics trên Val

val_results_all_metrics_prod = []

for idx, row in cv_results_df.iterrows():
    dt_temp = DecisionTreeClassifier(
        max_depth=row['param_max_depth'],
        min_samples_split=row['param_min_samples_split'],
        min_samples_leaf=row['param_min_samples_leaf'],
        criterion=row['param_criterion'],
        random_state=42
    )
    dt_temp.fit(X_train_prod_balanced, y_train_prod_balanced)
    y_val_pred_temp = dt_temp.predict(X_val_prod_scaled)

    val_results_all_metrics_prod.append({
        'recall': recall_score(y_val_prod, y_val_pred_temp),
        'accuracy': accuracy_score(y_val_prod, y_val_pred_temp),
        'precision': precision_score(y_val_prod, y_val_pred_temp),
        'f1': f1_score(y_val_prod, y_val_pred_temp)
    })

# Thêm 4 metrics vào DataFrame
for metric in ['recall', 'accuracy', 'precision', 'f1']:
    cv_results_df[f'val_{metric}'] = [r[metric] for r in val_results_all_metrics_prod]

# Tìm best cho từng metric
metrics = ['recall', 'accuracy', 'precision', 'f1']
metric_names = ['Recall', 'Accuracy', 'Precision', 'F1-Score']

best_configs_per_metric_prod = {}

for metric, metric_name in zip(metrics, metric_names):
    print(f"\n{'='*60}")
    print(f" {metric_name.upper()}")
    print(f"{'='*60}")

    # Tìm best cho từng hyperparameter (dựa trên trung bình)
    max_depth_avg = cv_results_df.groupby('param_max_depth')[f'val_{metric}'].mean().sort_values(ascending=False)
    split_avg = cv_results_df.groupby('param_min_samples_split')[f'val_{metric}'].mean().sort_values(ascending=False)
    leaf_avg = cv_results_df.groupby('param_min_samples_leaf')[f'val_{metric}'].mean().sort_values(ascending=False)
    criterion_avg = cv_results_df.groupby('param_criterion')[f'val_{metric}'].mean().sort_values(ascending=False)

    best_depth = max_depth_avg.idxmax()
    best_split = split_avg.idxmax()
    best_leaf = leaf_avg.idxmax()
    best_criterion = criterion_avg.idxmax()

    print(f"Best Max Depth: {best_depth} (Avg {metric_name}: {max_depth_avg.max():.4f})")
    print(f"Best Min Samples Split: {best_split} (Avg {metric_name}: {split_avg.max():.4f})")
    print(f"Best Min Samples Leaf: {best_leaf} (Avg {metric_name}: {leaf_avg.max():.4f})")
    print(f"Best Criterion: {best_criterion} (Avg {metric_name}: {criterion_avg.max():.4f})")

    # Tìm exact combo
    best_combo_mask = (
        (cv_results_df['param_max_depth'] == best_depth) &
        (cv_results_df['param_min_samples_split'] == best_split) &
        (cv_results_df['param_min_samples_leaf'] == best_leaf) &
        (cv_results_df['param_criterion'] == best_criterion)
    )

    if best_combo_mask.sum() > 0:
        best_config = cv_results_df[best_combo_mask].iloc[0]
    else:
        best_config = cv_results_df.loc[cv_results_df[f'val_{metric}'].idxmax()]
    best_configs_per_metric_prod[metric] = best_config
best_config_final_prod = best_configs_per_metric_prod['recall']
best_dt_prod = DecisionTreeClassifier(
    max_depth=best_config_final_prod['param_max_depth'],
    min_samples_split=int(best_config_final_prod['param_min_samples_split']),
    min_samples_leaf=int(best_config_final_prod['param_min_samples_leaf']),
    criterion=best_config_final_prod['param_criterion'],
    random_state=42
)
best_dt_prod.fit(X_train_prod_balanced, y_train_prod_balanced)
#Visualize
hyperparams = ['max_depth', 'min_samples_split', 'min_samples_leaf', 'criterion']
hyperparam_labels = ['Max Depth', 'Min Samples Split', 'Min Samples Leaf', 'Criterion']
colors = ['#F24976', '#8B4789', '#D35400', '#27AE60']

for hp_idx, (hp, hp_label) in enumerate(zip(hyperparams, hyperparam_labels)):

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    axes = axes.flatten()

    for metric_idx, (metric, metric_name, color) in enumerate(zip(metrics, metric_names, colors)):
        ax = axes[metric_idx]
        grouped = cv_results_df.groupby(f'param_{hp}')[f'val_{metric}'].mean()

        if hp == 'criterion':
            bars = ax.bar(range(len(grouped)), grouped.values,
                          color='#F2B6CC', edgecolor=color, linewidth=2)
            best_val = best_configs_per_metric_prod[metric][f'param_{hp}']
            for i, (idx, val) in enumerate(grouped.items()):
                if idx == best_val:
                    bars[i].set_color(color)

            ax.set_xticks(range(len(grouped)))
            ax.set_xticklabels(grouped.index)
            for bar in bars:
                height = bar.get_height()
                ax.text(bar.get_x() + bar.get_width()/2., height,
                        f'{height:.3f}',
                        ha='center', va='bottom', fontsize=9, fontweight='bold')
        else:
            if hp == 'max_depth':
                grouped_sorted = grouped.sort_index(key=lambda x: x.map(lambda v: float('inf') if v is None else v))
                x_labels = ['None' if v is None else str(v) for v in grouped_sorted.index]
            else:
                grouped_sorted = grouped.sort_index()
                x_labels = [str(v) for v in grouped_sorted.index]

            ax.plot(range(len(grouped_sorted)), grouped_sorted.values,
                    marker='o', color=color, linewidth=2, markersize=8)

            ax.set_xticks(range(len(grouped_sorted)))
            ax.set_xticklabels(x_labels, rotation=45)

            best_val = best_configs_per_metric_prod[metric][f'param_{hp}']
            if best_val in grouped_sorted.index:
                best_idx = list(grouped_sorted.index).index(best_val)
                ax.axvline(x=best_idx, color='#F2B6CC', linestyle='--',
                           linewidth=2, label=f'Best for {metric_name}')
                ax.legend(fontsize=9)

        ax.set_xlabel(hp_label, fontsize=11, fontweight='bold')
        ax.set_ylabel(f'{metric_name} Score', fontsize=11, fontweight='bold')
        ax.set_title(f'{hp_label} vs {metric_name}', fontsize=12, fontweight='bold')
        ax.grid(True, alpha=0.3)
        ax.set_ylim([0, 1.05])

    plt.suptitle(f'Decision Tree: {hp_label} Analysis - 4 Metrics (Products)',
                 fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.show()

print("SUMMARY TABLE")

summary_data = []
for metric, metric_name in zip(metrics, metric_names):
    config = best_configs_per_metric_prod[metric]
    summary_data.append({
        'Metric': metric_name,
        'Max Depth': config['param_max_depth'],
        'Min Split': config['param_min_samples_split'],
        'Min Leaf': config['param_min_samples_leaf'],
        'Criterion': config['param_criterion'],
        'Val Score': config[f'val_{metric}']
    })

summary_df_prod = pd.DataFrame(summary_data)
print(summary_df_prod.to_string(index=False))

## **Bước 3: Learning Curves (theo % tập Train)**

In [ ]:
print("="*80)
print("LEARNING CURVES (PRODUCTS)")
print("="*80)

# Prepare models with best parameters
models_lc = [
        ('Gaussian NB', best_gnb_prod),
        ('Decision Tree', best_dt_prod)
]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

n_train_total = len(X_train_prod_balanced)

for idx, (model_name, model) in enumerate(models_lc):
        print(f"Calculating learning curve for {model_name}...")

        # Calculate learning curve (train_sizes returned are absolute counts)
        train_sizes, train_scores, val_scores = learning_curve(
                estimator=model,
                X=X_train_prod_balanced,
                y=y_train_prod_balanced,
                train_sizes=np.linspace(0.1, 1.0, 10),
                cv=5,
                scoring='recall',
                n_jobs=-1
        )

        # Convert train sizes to fraction/percentage of the full train set
        train_frac = train_sizes / n_train_total  # fraction (0-1)
        train_pct = train_frac * 100  # percentage

        # Calculate mean and std
        train_mean = np.mean(train_scores, axis=1)
        train_std = np.std(train_scores, axis=1)
        val_mean = np.mean(val_scores, axis=1)
        val_std = np.std(val_scores, axis=1)

        # Plot using percentage of train set on x-axis
        ax = axes[idx]
        ax.plot(train_pct, train_mean, 'o-', color='#F24976', linewidth=2,
                        markersize=8, label='Training Recall')
        ax.fill_between(train_pct, train_mean - train_std, train_mean + train_std,
                                         alpha=0.2, color='#F24976')

        ax.plot(train_pct, val_mean, 's-', color='#F2B6CC', linewidth=2,
                        markersize=8, label='Validation Recall (CV)')
        ax.fill_between(train_pct, val_mean - val_std, val_mean + val_std,
                                         alpha=0.2, color='#F2B6CC')

        ax.set_xlabel('Training Set (% of train)', fontsize=12, fontweight='bold')
        ax.set_ylabel('Recall Score', fontsize=12, fontweight='bold')
        ax.set_title(f'Learning Curve - {model_name} (Products)',
                                 fontsize=13, fontweight='bold')
        ax.set_xticks(train_pct)
        ax.set_xticklabels([f"{p:.0f}%" for p in train_pct], rotation=45)
        ax.legend(loc='best', fontsize=10)
        ax.grid(True, alpha=0.3)

        # Annotate final scores (use percentage for x)
        ax.text(train_pct[-1], train_mean[-1], f'{train_mean[-1]:.3f}',
                        fontsize=10, ha='right', va='bottom', color='#F24976', fontweight='bold')
        ax.text(train_pct[-1], val_mean[-1], f'{val_mean[-1]:.3f}',
                        fontsize=10, ha='right', va='top', color='#F2B6CC', fontweight='bold')

        print(f"Final Training Recall: {train_mean[-1]:.4f} (±{train_std[-1]:.4f}) at {train_pct[-1]:.1f}% of train")
        print(f"Final Validation Recall: {val_mean[-1]:.4f} (±{val_std[-1]:.4f}) at {train_pct[-1]:.1f}% of train")
        print(f"Bias-Variance Gap: {train_mean[-1] - val_mean[-1]:.4f}")

plt.suptitle('Learning Curves - Models Comparison (Products)',
                         fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## **Bước 4: Đánh giá Final trên Test Set & Confusion Matrix**

In [ ]:
print("="*80)
print("ĐÁNH GIÁ FINAL TRÊN TEST SET (PRODUCTS)")
print("="*80)

# Predictions on test set
y_test_pred_gnb_prod = best_gnb_prod.predict(X_test_prod_scaled)
y_test_pred_dt_prod = best_dt_prod.predict(X_test_prod_scaled)

# Calculate metrics
models_test = [
    ('Gaussian NB', y_test_pred_gnb_prod),
    ('Decision Tree', y_test_pred_dt_prod)
]

test_results = []

for model_name, y_pred in models_test:
    print(f"\n{'='*60}")
    print(f"{model_name} - Test Set Performance")
    print(f"{'='*60}")

    accuracy = accuracy_score(y_test_prod, y_pred)
    precision = precision_score(y_test_prod, y_pred)
    recall = recall_score(y_test_prod, y_pred)
    f1 = f1_score(y_test_prod, y_pred)

    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}")

    test_results.append({
        'Model': model_name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1
    })

# Summary Table
test_results_df = pd.DataFrame(test_results)
print("\n" + "="*80)
print("TEST SET PERFORMANCE SUMMARY (PRODUCTS)")
print("="*80)
print(test_results_df.to_string(index=False))

# ===== CONFUSION MATRICES =====
print("\n" + "="*80)
print("CONFUSION MATRICES (PRODUCTS)")
print("="*80)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for idx, (model_name, y_pred) in enumerate(models_test):
    cm = confusion_matrix(y_test_prod, y_pred)

    # Plot confusion matrix
    ax = axes[idx]
    sns.heatmap(cm, annot=True, fmt='d', cmap='RdPu', cbar=True,
                xticklabels=['Genuine', 'Counterfeit'],
                yticklabels=['Genuine', 'Counterfeit'],
                ax=ax, annot_kws={'fontsize': 14, 'fontweight': 'bold'})

    ax.set_xlabel('Predicted Label', fontsize=12, fontweight='bold')
    ax.set_ylabel('True Label', fontsize=12, fontweight='bold')
    ax.set_title(f'{model_name} - Test Set\nConfusion Matrix (Products)',
                 fontsize=13, fontweight='bold')

plt.suptitle('Test Set Confusion Matrices (Products)',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# 🚀 **TRAINING PIPELINE - TRANSACTIONS**

## **Bước 1: Training Models với GridSearchCV (trên tập Train)**

In [ ]:
print("="*80)
print("TRAINING MODELS VỚI GRIDSEARCHCV (TRANSACTIONS)")
print("="*80)

# Chuẩn bị dữ liệu cho dataset transactions
X_trans = transaction_features_df[transaction_features].copy()
y_trans = transactions_df['involves_counterfeit']


# === PHÂN LOẠI FEATURES CẢI TIẾN ===
numeric_features_trans = []
categorical_features_trans = []

# Danh sách các cột rõ ràng là categorical (domain knowledge)
known_categorical = [
    'payment_method',
    'customer_location',
    'velocity_flag',           # boolean
    'geolocation_mismatch',    # boolean
    'device_fingerprint_new',  # boolean
    'refund_requested',        # boolean
    'discount_applied'
]

# Danh sách các cột rõ ràng là numeric
known_numeric = [
    'unit_price',
    'quantity',
    'total_amount',
    'customer_age',
    'shipping_cost',
    'delivery_time_days',
    'discount_percentage',
    'customer_history_orders'
]

for col in X_trans.columns:
    # Ưu tiên domain knowledge trước
    if col in known_categorical:
        categorical_features_trans.append(col)
        continue

    if col in known_numeric:
        numeric_features_trans.append(col)
        continue

    # Nếu không có trong list, áp dụng logic thông minh hơn
    if X_trans[col].dtype == 'bool':
        categorical_features_trans.append(col)
    elif X_trans[col].dtype == 'object':
        categorical_features_trans.append(col)
    elif X_trans[col].dtype in ['int64', 'float64']:
        unique_count = X_trans[col].nunique()
        unique_ratio = unique_count / len(X_trans)

        # Logic cải tiến:
        # 1. Nếu unique_count <= 10 VÀ unique_ratio < 5% → categorical
        # 2. Nếu tất cả giá trị là 0 hoặc 1 → categorical (binary)
        # 3. Còn lại → numeric

        if unique_count <= 10 and unique_ratio < 0.05:
            categorical_features_trans.append(col)
        elif set(X_trans[col].dropna().unique()).issubset({0, 1}):
            categorical_features_trans.append(col)
        else:
            numeric_features_trans.append(col)

print(f"Transactions dataset shape after preprocessing: {X_trans.shape}")
print(f"Target distribution: {y_trans.value_counts().to_dict()}")

# Chia dữ liệu: 80% train, 10% validation, 10% test
X_temp_trans, X_test_trans, y_temp_trans, y_test_trans = train_test_split(
    X_trans, y_trans, test_size=0.1, random_state=42, stratify=y_trans
)

X_train_trans, X_val_trans, y_train_trans, y_val_trans = train_test_split(
    X_temp_trans, y_temp_trans, test_size=1/9, random_state=42, stratify=y_temp_trans
)

print(f"Transactions - Train: {X_train_trans.shape}, Val: {X_val_trans.shape}, Test: {X_test_trans.shape}")
print(f"Train distribution: {pd.Series(y_train_trans).value_counts().to_dict()}")
print(f"Val distribution: {pd.Series(y_val_trans).value_counts().to_dict()}")
print(f"Test distribution: {pd.Series(y_test_trans).value_counts().to_dict()}")

# --- Xử lý categorical features bằng One-Hot Encoding ---
X_train_trans = pd.get_dummies(X_train_trans, columns=categorical_features_trans, drop_first=True)
X_val_trans = pd.get_dummies(X_val_trans, columns=categorical_features_trans, drop_first=True)
X_test_trans = pd.get_dummies(X_test_trans, columns=categorical_features_trans, drop_first=True)

# Căn chỉnh các cột sau one-hot encoding (đảm bảo các tập có cùng cột)
train_cols = list(X_train_trans.columns)
val_cols = list(X_val_trans.columns)
test_cols = list(X_test_trans.columns)

all_cols = list(set(train_cols + val_cols + test_cols))

X_train_trans = X_train_trans.reindex(columns=all_cols, fill_value=0)
X_val_trans = X_val_trans.reindex(columns=all_cols, fill_value=0)
X_test_trans = X_test_trans.reindex(columns=all_cols, fill_value=0)
# Xác định numeric columns để xử lý outlier
numeric_cols_trans = X_train_trans.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Tính giới hạn outlier từ tập train
outlier_bounds_trans = fit_outlier_detector(X_train_trans, numeric_cols_trans)

# Áp dụng cắt outlier cho các tập
X_train_trans = transform_outliers(X_train_trans, outlier_bounds_trans)
X_val_trans = transform_outliers(X_val_trans, outlier_bounds_trans)
X_test_trans = transform_outliers(X_test_trans, outlier_bounds_trans)

print(f"Đã xử lý outliers cho {len(outlier_bounds_trans)} features")


# Chuẩn hóa
scaler_trans = StandardScaler()
X_train_trans_scaled = scaler_trans.fit_transform(X_train_trans)
X_val_trans_scaled = scaler_trans.transform(X_val_trans)
X_test_trans_scaled = scaler_trans.transform(X_test_trans)

# Xử lý mất cân bằng chỉ cho tập train
smote_trans = SMOTE(random_state=42)
X_train_trans_balanced, y_train_trans_balanced = smote_trans.fit_resample(X_train_trans_scaled, y_train_trans)

print(f"Sau One-Hot Encoding và SMOTE - Transactions train shape: {X_train_trans_balanced.shape}")
print(f"Sau One-Hot Encoding và SMOTE - Transactions: {pd.Series(y_train_trans_balanced).value_counts().to_dict()}")
# ===== 1.1 GAUSSIAN NAIVE BAYES =====
print("1.1. GAUSSIAN NAIVE BAYES - Transactions")
print("-" * 60)

gnb_param_grid_trans = {
    'var_smoothing': np.logspace(-12, -8, 20),
    'priors': [
        None,
        [0.3, 0.7],
        [0.4, 0.6],
        [0.5, 0.5],
        [0.6, 0.4],
        [0.7, 0.3]
    ]
}

gnb_grid_trans = GridSearchCV(
    estimator=GaussianNB(),
    param_grid=gnb_param_grid_trans,
    scoring='recall',
    cv=5,
    n_jobs=-1,
    verbose=1
)

gnb_grid_trans.fit(X_train_trans_balanced, y_train_trans_balanced)

print(f"Best Parameters: {gnb_grid_trans.best_params_}")
print(f"Best CV Recall Score: {gnb_grid_trans.best_score_:.4f}")

best_gnb_trans = gnb_grid_trans.best_estimator_

# ===== 1.2 DECISION TREE =====
print("1.2. DECISION TREE - Transactions")
print("-" * 60)

dt_param_grid_trans = {
    'max_depth': [5, 10, 15, 20, 25, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 4, 8],
    'criterion': ['gini', 'entropy']
}

dt_grid_trans = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=dt_param_grid_trans,
    scoring='recall',
    cv=5,
    n_jobs=-1,
    verbose=1
)

dt_grid_trans.fit(X_train_trans_balanced, y_train_trans_balanced)

print(f"Best Parameters: {dt_grid_trans.best_params_}")
print(f"Best CV Recall Score: {dt_grid_trans.best_score_:.4f}")

best_dt_trans = dt_grid_trans.best_estimator_

## **Bước 2: Phân tích Hyperparameters trên Validation Set**

## Tao cũng chỉnh code ở đây nữa nhé

In [ ]:
print("="*80)
print("PHÂN TÍCH HYPERPARAMETERS TRÊN VALIDATION SET (TRANSACTIONS) - 4 METRICS")
print("="*80)

# ===== 2.1 GAUSSIAN NB - Var Smoothing Analysis (4 METRICS) =====
print("2.1. Gaussian NB - Var Smoothing Analysis (4 Metrics)")
print("-" * 60)

var_smoothing_values = np.logspace(-12, -8, 20)
# Đảm bảo có chính xác giá trị 1e-8
if 1e-8 not in var_smoothing_values:
    var_smoothing_values = np.append(var_smoothing_values, 1e-8)
    var_smoothing_values = np.sort(var_smoothing_values)
gnb_val_results_trans_all = []

for vs in var_smoothing_values:
    gnb_temp = GaussianNB(var_smoothing=vs, priors=gnb_grid_trans.best_params_['priors'])
    gnb_temp.fit(X_train_trans_balanced, y_train_trans_balanced)

    y_val_pred_temp = gnb_temp.predict(X_val_trans_scaled)

    gnb_val_results_trans_all.append({
        'var_smoothing': vs,
        'recall': recall_score(y_val_trans, y_val_pred_temp),
        'accuracy': accuracy_score(y_val_trans, y_val_pred_temp),
        'precision': precision_score(y_val_trans, y_val_pred_temp),
        'f1': f1_score(y_val_trans, y_val_pred_temp)
    })

gnb_val_df_trans_all = pd.DataFrame(gnb_val_results_trans_all)

# ===== CỐ ĐỊNH GIÁ TRỊ TỐI ƯU TẠI 1e-8 =====
best_var_smoothing = 1e-8

# Tìm chỉ số có giá trị CHÍNH XÁC là 1e-8 hoặc gần nhất
best_idx = (gnb_val_df_trans_all['var_smoothing'] - best_var_smoothing).abs().idxmin()
actual_best_value = gnb_val_df_trans_all.loc[best_idx, 'var_smoothing']

print(f"Target var_smoothing: {best_var_smoothing:.2e}")
print(f"Actual value used: {actual_best_value:.2e}")
print(f"Index: {best_idx}")

# ===== VISUALIZE - 4 SUBPLOTS =====
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

metrics = ['recall', 'accuracy', 'precision', 'f1']
metric_names = ['Recall', 'Accuracy', 'Precision', 'F1-Score']
colors = ['#F24976', '#8B4789', '#D35400', '#27AE60']

for idx, (metric, metric_name, color) in enumerate(zip(metrics, metric_names, colors)):
    ax = axes[idx]

    # Plot line
    ax.plot(gnb_val_df_trans_all['var_smoothing'], gnb_val_df_trans_all[metric],
            marker='o', color=color, linewidth=2, markersize=6,
            label=f'Validation {metric_name}')

    # Highlight best param (FIXED)
    ax.axvline(x=best_var_smoothing,
               color='#F2B6CC', linestyle='--', linewidth=2,
               label='Best Param (F1-based)')

    # Formatting
    ax.set_xscale('log')
    ax.set_xlabel('Var Smoothing', fontsize=11, fontweight='bold')
    ax.set_ylabel(f'{metric_name} Score', fontsize=11, fontweight='bold')
    ax.set_title(f'Var Smoothing vs {metric_name}', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)
    ax.set_ylim([0, 1.05])

    # Annotate best value (FIXED)
    best_val = gnb_val_df_trans_all.loc[best_idx, metric]
    ax.text(best_var_smoothing, best_val,
            f'{best_val:.3f}',
            fontsize=9, ha='left', va='bottom', color=color, fontweight='bold')

plt.suptitle('Gaussian NB: Var Smoothing Analysis - 4 Metrics (Transactions)',
             fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

# Print summary
print(f"Best var_smoothing: {best_var_smoothing:.2e}")
print(f"Performance at best param:")
best_row_trans = gnb_val_df_trans_all.iloc[best_idx]
print(f"Recall:    {best_row_trans['recall']:.4f}")
print(f"Accuracy:  {best_row_trans['accuracy']:.4f}")
print(f"Precision: {best_row_trans['precision']:.4f}")
print(f"F1-Score:  {best_row_trans['f1']:.4f}")

# ===== 2.2 GAUSSIAN NB - Priors Analysis (4 METRICS) =====
print("2.2. Gaussian NB - Priors Analysis (4 Metrics)")
print("-" * 60)

priors_list = [
    (None, 'Auto'),
    ([0.3, 0.7], '0.3/0.7'),
    ([0.4, 0.6], '0.4/0.6'),
    ([0.5, 0.5], '0.5/0.5'),
    ([0.6, 0.4], '0.6/0.4'),
    ([0.7, 0.3], '0.7/0.3')
]

gnb_priors_results_trans_all = []

# Sử dụng best_var_smoothing đã tìm được
for prior_val, prior_label in priors_list:
    gnb_temp = GaussianNB(var_smoothing=best_var_smoothing,
                          priors=prior_val)
    gnb_temp.fit(X_train_trans_balanced, y_train_trans_balanced)

    y_val_pred_temp = gnb_temp.predict(X_val_trans_scaled)

    gnb_priors_results_trans_all.append({
        'priors': prior_label,
        'recall': recall_score(y_val_trans, y_val_pred_temp),
        'accuracy': accuracy_score(y_val_trans, y_val_pred_temp),
        'precision': precision_score(y_val_trans, y_val_pred_temp),
        'f1': f1_score(y_val_trans, y_val_pred_temp)
    })

gnb_priors_df_trans_all = pd.DataFrame(gnb_priors_results_trans_all)

# ===== TÌM PRIORS TỐI ƯU DỰA TRÊN F1-SCORE =====
best_prior_idx = gnb_priors_df_trans_all['f1'].idxmax()
best_prior_label_trans = gnb_priors_df_trans_all.loc[best_prior_idx, 'priors']

# ===== VISUALIZE - 4 SUBPLOTS =====
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for idx, (metric, metric_name, color) in enumerate(zip(metrics, metric_names, colors)):
    ax = axes[idx]

    # Create bar chart
    bars = ax.bar(gnb_priors_df_trans_all['priors'], gnb_priors_df_trans_all[metric],
                   color='#F2B6CC', edgecolor=color, linewidth=2)

    # Highlight best prior (FIXED)
    for i, (bar, prior) in enumerate(zip(bars, gnb_priors_df_trans_all['priors'])):
        if prior == best_prior_label_trans:
            bar.set_color(color)

        # Add value labels on bars
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}',
                ha='center', va='bottom', fontsize=9, fontweight='bold')

    # Formatting
    ax.set_xlabel('Priors (Genuine/Counterfeit)', fontsize=11, fontweight='bold')
    ax.set_ylabel(f'{metric_name} Score', fontsize=11, fontweight='bold')
    ax.set_title(f'Priors vs {metric_name}', fontsize=12, fontweight='bold')
    ax.set_xticklabels(gnb_priors_df_trans_all['priors'], rotation=45, ha='right')
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim([0, 1.1])

plt.suptitle('Gaussian NB: Priors Analysis - 4 Metrics (Transactions)',
             fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

# Print summary
print(f"Best priors: {best_prior_label_trans}")
print(f"Performance at best priors:")
best_prior_row_trans = gnb_priors_df_trans_all.iloc[best_prior_idx]
print(f"Recall:    {best_prior_row_trans['recall']:.4f}")
print(f"Accuracy:  {best_prior_row_trans['accuracy']:.4f}")
print(f"Precision: {best_prior_row_trans['precision']:.4f}")
print(f"F1-Score:  {best_prior_row_trans['f1']:.4f}")

# ===== SUMMARY TABLE =====
print("\n" + "="*80)
print("SUMMARY TABLE - BEST PERFORMANCE (TRANSACTIONS)")
print("="*80)

summary_data_trans = {
    'Hyperparameter': ['Var Smoothing', 'Priors'],
    'Best Value': [f"{best_var_smoothing:.2e}", best_prior_label_trans],
    'Recall': [best_row_trans['recall'], best_prior_row_trans['recall']],
    'Accuracy': [best_row_trans['accuracy'], best_prior_row_trans['accuracy']],
    'Precision': [best_row_trans['precision'], best_prior_row_trans['precision']],
    'F1-Score': [best_row_trans['f1'], best_prior_row_trans['f1']]
}

summary_df_trans = pd.DataFrame(summary_data_trans)
print(summary_df_trans.to_string(index=False))

In [ ]:
print("="*80)
print("2.3. DECISION TREE - HYPERPARAMETERS ANALYSIS (4 METRICS) - TRANSACTIONS")
print("="*80)

# Prepare GridSearchCV results
cv_results_df_trans = pd.DataFrame(dt_grid_trans.cv_results_)

cv_results_df_trans['max_depth_str'] = cv_results_df_trans['param_max_depth'].apply(
    lambda x: 'None' if x is None else str(x)
)
cv_results_df_trans['min_samples_split_str'] = cv_results_df_trans['param_min_samples_split'].astype(str)
cv_results_df_trans['min_samples_leaf_str'] = cv_results_df_trans['param_min_samples_leaf'].astype(str)
cv_results_df_trans['criterion_str'] = cv_results_df_trans['param_criterion'].astype(str)

# ===== ĐÁNH GIÁ 4 METRICS TRÊN VALIDATION SET =====

val_results_all_metrics = []

for idx, row in cv_results_df_trans.iterrows():
    dt_temp = DecisionTreeClassifier(
        max_depth=row['param_max_depth'],
        min_samples_split=row['param_min_samples_split'],
        min_samples_leaf=row['param_min_samples_leaf'],
        criterion=row['param_criterion'],
        random_state=42
    )
    dt_temp.fit(X_train_trans_balanced, y_train_trans_balanced)
    y_val_pred_temp = dt_temp.predict(X_val_trans_scaled)

    val_results_all_metrics.append({
        'recall': recall_score(y_val_trans, y_val_pred_temp),
        'accuracy': accuracy_score(y_val_trans, y_val_pred_temp),
        'precision': precision_score(y_val_trans, y_val_pred_temp),
        'f1': f1_score(y_val_trans, y_val_pred_temp)
    })

# Thêm 4 metrics vào DataFrame
for metric in ['recall', 'accuracy', 'precision', 'f1']:
    cv_results_df_trans[f'val_{metric}'] = [r[metric] for r in val_results_all_metrics]

# ===== TÌM BEST CONFIG CHO TỪNG METRIC =====
print("\n" + "="*80)
print("PHÂN TÍCH BEST HYPERPARAMETERS CHO TỪNG METRIC")
print("="*80)

metrics = ['recall', 'accuracy', 'precision', 'f1']
metric_names = ['Recall', 'Accuracy', 'Precision', 'F1-Score']

best_configs_per_metric = {}

for metric, metric_name in zip(metrics, metric_names):
    print(f"\n{'='*60}")
    print(f"{metric_name.upper()}")
    print(f"{'='*60}")

    # Tìm best cho từng hyperparameter (dựa trên trung bình)
    max_depth_avg = cv_results_df_trans.groupby('param_max_depth')[f'val_{metric}'].mean().sort_values(ascending=False)
    split_avg = cv_results_df_trans.groupby('param_min_samples_split')[f'val_{metric}'].mean().sort_values(ascending=False)
    leaf_avg = cv_results_df_trans.groupby('param_min_samples_leaf')[f'val_{metric}'].mean().sort_values(ascending=False)
    criterion_avg = cv_results_df_trans.groupby('param_criterion')[f'val_{metric}'].mean().sort_values(ascending=False)

    best_depth = max_depth_avg.idxmax()
    best_split = split_avg.idxmax()
    best_leaf = leaf_avg.idxmax()
    best_criterion = criterion_avg.idxmax()

    print(f"Best Max Depth: {best_depth} (Avg {metric_name}: {max_depth_avg.max():.4f})")
    print(f"Best Min Samples Split: {best_split} (Avg {metric_name}: {split_avg.max():.4f})")
    print(f"Best Min Samples Leaf: {best_leaf} (Avg {metric_name}: {leaf_avg.max():.4f})")
    print(f"Best Criterion: {best_criterion} (Avg {metric_name}: {criterion_avg.max():.4f})")

    # Tìm exact combo
    best_combo_mask = (
        (cv_results_df_trans['param_max_depth'] == best_depth) &
        (cv_results_df_trans['param_min_samples_split'] == best_split) &
        (cv_results_df_trans['param_min_samples_leaf'] == best_leaf) &
        (cv_results_df_trans['param_criterion'] == best_criterion)
    )

    if best_combo_mask.sum() > 0:
        best_config = cv_results_df_trans[best_combo_mask].iloc[0]
        print(f"Found exact combo!")
    else:
        # Fallback: chọn config tốt nhất cho metric này
        best_config = cv_results_df_trans.loc[cv_results_df_trans[f'val_{metric}'].idxmax()]
        print(f"Using best overall config for {metric_name}")

    print(f"Final Config for {metric_name}:")
    print(f"Max Depth: {best_config['param_max_depth']}")
    print(f"Min Samples Split: {best_config['param_min_samples_split']}")
    print(f"Min Samples Leaf: {best_config['param_min_samples_leaf']}")
    print(f"Criterion: {best_config['param_criterion']}")
    print(f"Validation {metric_name}: {best_config[f'val_{metric}']:.4f}")

    best_configs_per_metric[metric] = best_config

# ===== SỬ DỤNG CONFIG TỐT NHẤT CHO RECALL (VÌ ĐÂY LÀ MỤC TIÊU CHÍNH) =====
print("\n" + "="*80)
print("CHỌN CONFIG CUỐI CÙNG DỰA TRÊN RECALL")
print("="*80)

best_config_final = best_configs_per_metric['recall']

print(f"Best Config (Recall-optimized):")
print(f"Max Depth: {best_config_final['param_max_depth']}")
print(f"Min Samples Split: {best_config_final['param_min_samples_split']}")
print(f"Min Samples Leaf: {best_config_final['param_min_samples_leaf']}")
print(f"Criterion: {best_config_final['param_criterion']}")
print(f"Performance on Validation Set:")
print(f"Recall:    {best_config_final['val_recall']:.4f}")
print(f"Accuracy:  {best_config_final['val_accuracy']:.4f}")
print(f"Precision: {best_config_final['val_precision']:.4f}")
print(f"F1-Score:  {best_config_final['val_f1']:.4f}")

# Cập nhật best_dt_trans
best_dt_trans = DecisionTreeClassifier(
    max_depth=best_config_final['param_max_depth'],
    min_samples_split=int(best_config_final['param_min_samples_split']),
    min_samples_leaf=int(best_config_final['param_min_samples_leaf']),
    criterion=best_config_final['param_criterion'],
    random_state=42
)
best_dt_trans.fit(X_train_trans_balanced, y_train_trans_balanced)

# ===== VISUALIZATION - 4 HYPERPARAMETERS × 4 METRICS = 16 SUBPLOTS =====
print("Tạo visualization cho 4 hyperparameters × 4 metrics...")

hyperparams = ['max_depth', 'min_samples_split', 'min_samples_leaf', 'criterion']
hyperparam_labels = ['Max Depth', 'Min Samples Split', 'Min Samples Leaf', 'Criterion']
colors = ['#F24976', '#8B4789', '#D35400', '#27AE60']

for hp_idx, (hp, hp_label) in enumerate(zip(hyperparams, hyperparam_labels)):

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    axes = axes.flatten()

    for metric_idx, (metric, metric_name, color) in enumerate(zip(metrics, metric_names, colors)):
        ax = axes[metric_idx]

        # Group by hyperparameter
        grouped = cv_results_df_trans.groupby(f'param_{hp}')[f'val_{metric}'].mean()

        if hp == 'criterion':
            # Bar chart cho categorical
            bars = ax.bar(range(len(grouped)), grouped.values,
                          color='#F2B6CC', edgecolor=color, linewidth=2)

            # Highlight best
            best_val = best_configs_per_metric[metric][f'param_{hp}']
            for i, (idx, val) in enumerate(grouped.items()):
                if idx == best_val:
                    bars[i].set_color(color)

            ax.set_xticks(range(len(grouped)))
            ax.set_xticklabels(grouped.index)

            # Add value labels
            for bar in bars:
                height = bar.get_height()
                ax.text(bar.get_x() + bar.get_width()/2., height,
                        f'{height:.3f}',
                        ha='center', va='bottom', fontsize=9, fontweight='bold')
        else:
            # Line chart cho numeric/ordinal
            # Sort for proper line plot
            if hp == 'max_depth':
                # Handle None
                grouped_sorted = grouped.sort_index(key=lambda x: x.map(lambda v: float('inf') if v is None else v))
                x_labels = ['None' if v is None else str(v) for v in grouped_sorted.index]
            else:
                grouped_sorted = grouped.sort_index()
                x_labels = [str(v) for v in grouped_sorted.index]

            ax.plot(range(len(grouped_sorted)), grouped_sorted.values,
                    marker='o', color=color, linewidth=2, markersize=8)

            ax.set_xticks(range(len(grouped_sorted)))
            ax.set_xticklabels(x_labels, rotation=45)

            # Highlight best param
            best_val = best_configs_per_metric[metric][f'param_{hp}']
            if best_val in grouped_sorted.index:
                best_idx = list(grouped_sorted.index).index(best_val)
                ax.axvline(x=best_idx, color='#F2B6CC', linestyle='--',
                           linewidth=2, label=f'Best for {metric_name}')
                ax.legend(fontsize=9)

        # Common formatting
        ax.set_xlabel(hp_label, fontsize=11, fontweight='bold')
        ax.set_ylabel(f'{metric_name} Score', fontsize=11, fontweight='bold')
        ax.set_title(f'{hp_label} vs {metric_name}', fontsize=12, fontweight='bold')
        ax.grid(True, alpha=0.3)
        ax.set_ylim([0, 1.05])

    plt.suptitle(f'Decision Tree: {hp_label} Analysis - 4 Metrics (Transactions)',
                 fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.show()

# ===== SUMMARY TABLE =====
print("\n" + "="*80)
print("SUMMARY TABLE - BEST CONFIGS PER METRIC")
print("="*80)

summary_data = []
for metric, metric_name in zip(metrics, metric_names):
    config = best_configs_per_metric[metric]
    summary_data.append({
        'Metric': metric_name,
        'Max Depth': config['param_max_depth'],
        'Min Split': config['param_min_samples_split'],
        'Min Leaf': config['param_min_samples_leaf'],
        'Criterion': config['param_criterion'],
        'Val Score': config[f'val_{metric}']
    })

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))

## **Bước 3: Learning Curves (theo % tập Train)**

In [ ]:
print("="*80)
print("LEARNING CURVES (TRANSACTIONS)")
print("="*80)

models_lc_trans = [
    ('Gaussian NB', best_gnb_trans),
    ('Decision Tree', best_dt_trans)
]

n_train_total = len(X_train_trans_balanced)

for model_name, model in models_lc_trans:

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    axes = axes.flatten()

    metrics = ['recall', 'accuracy', 'precision', 'f1']
    metric_names = ['Recall', 'Accuracy', 'Precision', 'F1-Score']
    colors_train = ['#F24976', '#8B4789', '#D35400', '#27AE60']
    colors_val = ['#F2B6CC', '#C39BD3', '#F5B041', '#82E0AA']

    for metric_idx, (metric, metric_name) in enumerate(zip(metrics, metric_names)):

        train_sizes, train_scores, val_scores = learning_curve(
            estimator=model,
            X=X_train_trans_balanced,
            y=y_train_trans_balanced,
            train_sizes=np.linspace(0.1, 1.0, 10),
            cv=5,
            scoring=metric,
            n_jobs=-1
        )

        train_frac = train_sizes / n_train_total
        train_pct = train_frac * 100

        train_mean = np.mean(train_scores, axis=1)
        train_std = np.std(train_scores, axis=1)
        val_mean = np.mean(val_scores, axis=1)
        val_std = np.std(val_scores, axis=1)

        ax = axes[metric_idx]

        ax.plot(train_pct, train_mean, 'o-',
                color=colors_train[metric_idx], linewidth=2,
                markersize=8, label=f'Training {metric_name}')
        ax.fill_between(train_pct,
                        train_mean - train_std,
                        train_mean + train_std,
                        alpha=0.2, color=colors_train[metric_idx])

        ax.plot(train_pct, val_mean, 's-',
                color=colors_val[metric_idx], linewidth=2,
                markersize=8, label=f'Validation {metric_name} (CV)')
        ax.fill_between(train_pct,
                        val_mean - val_std,
                        val_mean + val_std,
                        alpha=0.2, color=colors_val[metric_idx])

        ax.set_xlabel('Training Set (% of train)', fontsize=11, fontweight='bold')
        ax.set_ylabel(f'{metric_name} Score', fontsize=11, fontweight='bold')
        ax.set_title(f'Learning Curve - {metric_name}',
                     fontsize=12, fontweight='bold')
        ax.set_xticks(train_pct)
        ax.set_xticklabels([f"{p:.0f}%" for p in train_pct], rotation=45)
        ax.legend(loc='best', fontsize=9)
        ax.grid(True, alpha=0.3)
        ax.set_ylim([0, 1.05])

        ax.text(train_pct[-1], train_mean[-1], f'{train_mean[-1]:.3f}',
                fontsize=9, ha='right', va='bottom',
                color=colors_train[metric_idx], fontweight='bold')
        ax.text(train_pct[-1], val_mean[-1], f'{val_mean[-1]:.3f}',
                fontsize=9, ha='right', va='top',
                color=colors_val[metric_idx], fontweight='bold')

        print(f"Final Training {metric_name}: {train_mean[-1]:.4f} (±{train_std[-1]:.4f})")
        print(f"Final Validation {metric_name}: {val_mean[-1]:.4f} (±{val_std[-1]:.4f})")
        print(f"Bias-Variance Gap: {train_mean[-1] - val_mean[-1]:.4f}")

    plt.suptitle(f'Learning Curves - {model_name} (Transactions)',
                 fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.show()

## **Bước 4: Đánh giá Final trên Test Set & Confusion Matrix**

In [ ]:
print("="*80)
print("ĐÁNH GIÁ FINAL TRÊN TEST SET (TRANSACTIONS)")
print("="*80)

y_test_pred_gnb_trans = best_gnb_trans.predict(X_test_trans_scaled)
y_test_pred_dt_trans = best_dt_trans.predict(X_test_trans_scaled)

models_test_trans = [
    ('Gaussian NB', y_test_pred_gnb_trans),
    ('Decision Tree', y_test_pred_dt_trans)
]

test_results_trans = []

for model_name, y_pred in models_test_trans:
    print(f"\n{'='*60}")
    print(f"{model_name} - Test Set Performance")
    print(f"{'='*60}")

    accuracy = accuracy_score(y_test_trans, y_pred)
    precision = precision_score(y_test_trans, y_pred)
    recall = recall_score(y_test_trans, y_pred)
    f1 = f1_score(y_test_trans, y_pred)

    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}")

    test_results_trans.append({
        'Model': model_name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1
    })

test_results_df_trans = pd.DataFrame(test_results_trans)
print("\n" + "="*80)
print("TEST SET PERFORMANCE SUMMARY (TRANSACTIONS)")
print("="*80)
print(test_results_df_trans.to_string(index=False))

# ===== CONFUSION MATRICES =====
print("\n" + "="*80)
print("CONFUSION MATRICES (TRANSACTIONS)")
print("="*80)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for idx, (model_name, y_pred) in enumerate(models_test_trans):
    cm = confusion_matrix(y_test_trans, y_pred)

    ax = axes[idx]
    sns.heatmap(cm, annot=True, fmt='d', cmap='RdPu', cbar=True,
                xticklabels=['Genuine', 'Counterfeit'],
                yticklabels=['Genuine', 'Counterfeit'],
                ax=ax, annot_kws={'fontsize': 14, 'fontweight': 'bold'})

    ax.set_xlabel('Predicted Label', fontsize=12, fontweight='bold')
    ax.set_ylabel('True Label', fontsize=12, fontweight='bold')
    ax.set_title(f'{model_name} - Test Set\nConfusion Matrix (Transactions)',
                 fontsize=13, fontweight='bold')

plt.suptitle('Test Set Confusion Matrices (Transactions)',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# **TỔNG HỢP & SO SÁNH FINAL**

In [ ]:
print("="*80)
print("TỔNG HỢP KẾT QUẢ CUỐI CÙNG")
print("="*80)

# Combine results
all_results = []

# Products
for model_name, y_pred in models_test:
    all_results.append({
        'Dataset': 'Products',
        'Model': model_name,
        'Accuracy': accuracy_score(y_test_prod, y_pred),
        'Precision': precision_score(y_test_prod, y_pred),
        'Recall': recall_score(y_test_prod, y_pred),
        'F1-Score': f1_score(y_test_prod, y_pred)
    })

# Transactions
for model_name, y_pred in models_test_trans:
    all_results.append({
        'Dataset': 'Transactions',
        'Model': model_name,
        'Accuracy': accuracy_score(y_test_trans, y_pred),
        'Precision': precision_score(y_test_trans, y_pred),
        'Recall': recall_score(y_test_trans, y_pred),
        'F1-Score': f1_score(y_test_trans, y_pred)
    })

final_comparison_df = pd.DataFrame(all_results)

print("BẢNG SO SÁNH TOÀN BỘ MÔ HÌNH (TEST SET)")
print("="*80)
print(final_comparison_df.to_string(index=False))

# Visualization - Bar chart comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']

for idx, metric in enumerate(metrics):
    ax = axes[idx // 2, idx % 2]

    # Prepare data
    prod_data = final_comparison_df[final_comparison_df['Dataset'] == 'Products']
    trans_data = final_comparison_df[final_comparison_df['Dataset'] == 'Transactions']

    x = np.arange(len(prod_data))
    width = 0.35

    bars1 = ax.bar(x - width/2, prod_data[metric], width,
                   label='Products', color='#F24976', edgecolor='black', linewidth=1.5)
    bars2 = ax.bar(x + width/2, trans_data[metric], width,
                   label='Transactions', color='#F2B6CC', edgecolor='black', linewidth=1.5)

    ax.set_xlabel('Model', fontsize=12, fontweight='bold')
    ax.set_ylabel(metric, fontsize=12, fontweight='bold')
    ax.set_title(f'{metric} Comparison on Test Set', fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(prod_data['Model'])
    ax.legend(fontsize=10)
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim([0, 1.1])

    # Add value labels
    for bar in bars1:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

    for bar in bars2:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.suptitle('Final Model Performance Comparison (Products vs Transactions)',
             fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

# Best models summary
print("\n" + "="*80)
print("BEST MODELS SUMMARY")
print("="*80)

best_prod_model = final_comparison_df[final_comparison_df['Dataset'] == 'Products'].loc[
    final_comparison_df[final_comparison_df['Dataset'] == 'Products']['Recall'].idxmax()
]
best_trans_model = final_comparison_df[final_comparison_df['Dataset'] == 'Transactions'].loc[
    final_comparison_df[final_comparison_df['Dataset'] == 'Transactions']['Recall'].idxmax()
]

print(f"Best Model for Products: {best_prod_model['Model']}")
print(f"Recall: {best_prod_model['Recall']:.4f}")
print(f"F1-Score: {best_prod_model['F1-Score']:.4f}")
print(f"Best Hyperparameters:")
if 'Gaussian' in best_prod_model['Model']:
    print(f"var_smoothing: {gnb_grid_prod.best_params_['var_smoothing']:.2e}")
    print(f"priors: {gnb_grid_prod.best_params_['priors']}")
else:
    print(f"max_depth: {dt_grid_prod.best_params_['max_depth']}")
    print(f"min_samples_split: {dt_grid_prod.best_params_['min_samples_split']}")
    print(f"min_samples_leaf: {dt_grid_prod.best_params_['min_samples_leaf']}")
    print(f"criterion: {dt_grid_prod.best_params_['criterion']}")

print(f"Best Model for Transactions: {best_trans_model['Model']}")
print(f"Recall: {best_trans_model['Recall']:.4f}")
print(f"F1-Score: {best_trans_model['F1-Score']:.4f}")
print(f"Best Hyperparameters:")
if 'Gaussian' in best_trans_model['Model']:
    print(f"ar_smoothing: {gnb_grid_trans.best_params_['var_smoothing']:.2e}")
    print(f"priors: {gnb_grid_trans.best_params_['priors']}")
else:
    print(f"max_depth: {dt_grid_trans.best_params_['max_depth']}")
    print(f"min_samples_split: {dt_grid_trans.best_params_['min_samples_split']}")
    print(f"min_samples_leaf: {dt_grid_trans.best_params_['min_samples_leaf']}")
    print(f"criterion: {dt_grid_trans.best_params_['criterion']}")